In [5]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 1: Portfolio Parsing
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# SPAN 2 differs from classic SPAN 1 in how it structures
# position data — it needs return series hooks, dollar-delta,
# and contract metadata all in one place from the start.
# ============================================================

import pandas as pd
import numpy as np
from datetime import date

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# ============================================================
# 1. RAW PORTFOLIO  (from FUTCOM_FINAL_PORTFOLIO.xlsx)
# ============================================================

portfolio_raw = {
    "Commodity"    : ["Turmeric","Coriander","Jeera","Guar Gum",
                      "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"       : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                      "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"   : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                      "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    # ↑ used in Step 2 for historical data fetch — update if
    # your data source uses different ticker conventions
    "Sector"       : ["Spices","Spices","Spices",
                      "Oilseeds & Derivatives","Oilseeds & Derivatives",
                      "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Expiry"       : ["20-Apr-2026","20-Apr-2026","20-Apr-2026","20-Apr-2026",
                      "20-Apr-2026","20-Apr-2026","20-Apr-2026","30-Apr-2026"],
    "Direction"    : ["LONG"] * 8,
    "Num_Lots"     : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_MT"  : [5, 5, 3, 5, 5, 5, 10, 4],
    "Multiplier"   : [50, 50, 30, 50, 50, 50, 100, 40],     # quintals per lot
    "Close_Rs_Qtl" : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val" : [811000, 667500, 668550, 537800,
                      285750, 319600, 356700, 67280],
}

df = pd.DataFrame(portfolio_raw)

# ============================================================
# 2. SPAN 2 POSITION ENRICHMENT
# ============================================================

TODAY = date(2026, 4, 8)

# --- 2a. Lot size in Quintals ---
df["Lot_Size_Qtl"] = df["Lot_Size_MT"] * 10

# --- 2b. Net Position & Direction Sign ---
#   +1 = Long,  −1 = Short
df["Direction_Sign"] = df["Direction"].map({"LONG": 1, "SHORT": -1})
df["Net_Lots"]       = df["Direction_Sign"] * df["Num_Lots"]

# --- 2c. Dollar Delta (₹ per 1% price move per lot) ---
# This is the SPAN 2 sensitivity unit:
# A 1% move in price moves the P&L by Dollar_Delta rupees
#   Dollar_Delta = Contract_Val × 0.01
df["Dollar_Delta_per_Lot"] = df["Contract_Val"] * 0.01
df["Dollar_Delta_Total"]   = df["Dollar_Delta_per_Lot"] * df["Num_Lots"]

# --- 2d. Total Position Value ---
df["Position_Value"] = df["Contract_Val"] * df["Num_Lots"]

# --- 2e. Contract Value Verification ---
df["CV_Check"] = df["Close_Rs_Qtl"] * df["Lot_Size_Qtl"]
df["CV_OK"]    = df["CV_Check"] == df["Contract_Val"]

# --- 2f. Days to Expiry ---
from datetime import datetime
df["Expiry_Date"]    = pd.to_datetime(df["Expiry"], format="%d-%b-%Y").dt.date
df["Days_to_Expiry"] = df["Expiry_Date"].apply(lambda d: (d - TODAY).days)

# ============================================================
# 3. SPAN 2 SPECIFIC METADATA
# ============================================================
# SPAN 2 uses a risk-factor approach:
#   - Each commodity futures price is a RISK FACTOR
#   - Margin = f(volatility of risk factor × dollar delta)
#   - Historical Simulation window = 252 trading days (1 year)
#   - Confidence level = 99% one-tailed
#   - Holding period = 1 trading day

SPAN2_CONFIG = {
    "confidence_level"  : 0.99,      # 99% VaR
    "holding_period"    : 1,         # 1 trading day
    "hs_window"         : 252,       # historical simulation lookback (days)
    "stress_window"     : 63,        # stress period window (quarter, ~3 months)
    "return_type"       : "log",     # log returns: ln(P_t / P_{t-1})
    "valuation_date"    : str(TODAY),
    "extreme_weight"    : 0.35,      # weight on stress VaR component
}

# ============================================================
# 4. DISPLAY
# ============================================================

print("=" * 72)
print("  SPAN 2 — STEP 1 : PORTFOLIO PARSING")
print("=" * 72)
print(f"\n  Valuation Date  : {TODAY}")
print(f"  Methodology     : Historical Simulation VaR @ {SPAN2_CONFIG['confidence_level']*100:.0f}%")
print(f"  Lookback Window : {SPAN2_CONFIG['hs_window']} trading days")
print(f"  Return Type     : {SPAN2_CONFIG['return_type'].capitalize()} returns")

print("\n\n📋 Position Table:")
pos_cols = ["Commodity", "Symbol", "Sector", "Direction",
            "Num_Lots", "Lot_Size_Qtl", "Close_Rs_Qtl",
            "Contract_Val", "Position_Value"]
print(df[pos_cols].to_string(index=False))

print("\n\n💰 Dollar Delta (₹ P&L per 1% price move):")
dd_cols = ["Commodity", "Symbol", "Net_Lots",
           "Dollar_Delta_per_Lot", "Dollar_Delta_Total"]
print(df[dd_cols].to_string(index=False))

print("\n\n📅 Expiry & Days to Expiry:")
exp_cols = ["Commodity", "Symbol", "Expiry", "Days_to_Expiry"]
print(df[exp_cols].to_string(index=False))

# Totals
total_pos_val  = df["Position_Value"].sum()
total_dd       = df["Dollar_Delta_Total"].sum()

print(f"\n{'─'*60}")
print(f"  Total Position Value         : ₹{total_pos_val:>14,.2f}")
print(f"  Total Dollar Delta (@ 1% mv) : ₹{total_dd:>14,.2f}")
print(f"  Number of Risk Factors       :  {len(df):>13d}")
print(f"{'─'*60}")

# --- CV Verification ---
print("\n\n✅ Contract Value Verification:")
print(df[["Commodity","CV_Check","Contract_Val","CV_OK"]].to_string(index=False))
all_ok = df["CV_OK"].all()
print(f"\n  All values verified: {'✅ YES' if all_ok else '❌ MISMATCH FOUND'}")

print("\n\n⚙️  SPAN 2 Configuration:")
for k, v in SPAN2_CONFIG.items():
    print(f"   {k:<22s}: {v}")

# ============================================================
# 5. OUTPUT  (passes to Step 2)
# ============================================================

span2_step1 = df[[
    "Commodity", "Symbol", "MCX_Ticker", "Sector", "Expiry",
    "Expiry_Date", "Days_to_Expiry",
    "Direction", "Direction_Sign", "Num_Lots", "Net_Lots",
    "Lot_Size_Qtl", "Close_Rs_Qtl",
    "Contract_Val", "Position_Value",
    "Dollar_Delta_per_Lot", "Dollar_Delta_Total",
]].copy()

print(f"\n\n🔁 `span2_step1` dataframe ready → passes into Step 2 (Historical Price Fetch & Return Series)")
print(f"   Shape: {span2_step1.shape}  |  Columns: {list(span2_step1.columns)}")

  SPAN 2 — STEP 1 : PORTFOLIO PARSING

  Valuation Date  : 2026-04-08
  Methodology     : Historical Simulation VaR @ 99%
  Lookback Window : 252 trading days
  Return Type     : Log returns


📋 Position Table:
           Commodity     Symbol                 Sector Direction  Num_Lots  Lot_Size_Qtl  Close_Rs_Qtl  Contract_Val  Position_Value
            Turmeric  TMCFGRNZM                 Spices      LONG         1            50         16220        811000          811000
           Coriander    DHANIYA                 Spices      LONG         2            50         13350        667500         1335000
               Jeera JEERAUNJHA                 Spices      LONG         2            30         22285        668550         1337100
            Guar Gum   GUARGUM5 Oilseeds & Derivatives      LONG         2            50         10756        537800         1075600
           Guar Seed GUARSEED10 Oilseeds & Derivatives      LONG         4            50          5715        285750        

In [6]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 2: Historical Price Fetch
#                                    & Log Return Series
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span2_step1  (DataFrame from Step 1)
#
# NOTE ON DATA SOURCE:
# MCX futures OHLC is NOT available on yfinance.
# Two approaches are provided — pick ONE:
#   [A] Upload your own MCX price CSV  (recommended for production)
#   [B] Synthetic series via GBM       (used here for demonstration)
#
# For live use: download 1-year daily close from
#   → MCX website → Market Data → Historical Data
#   → Stooq / Quandl / Bloomberg for MCX futures
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, timedelta, datetime
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# ── Reproducibility ─────────────────────────────────────────
np.random.seed(42)

# ============================================================
# PASTE span2_step1 HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"           : ["Turmeric","Coriander","Jeera","Guar Gum",
                              "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"              : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                              "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"          : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                              "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    "Sector"              : ["Spices","Spices","Spices",
                              "Oilseeds & Derivatives","Oilseeds & Derivatives",
                              "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Direction"           : ["LONG"] * 8,
    "Direction_Sign"      : [1] * 8,
    "Num_Lots"            : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Lots"            : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"        : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"        : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"        : [811000, 667500, 668550, 537800,
                              285750, 319600, 356700, 67280],
    "Position_Value"      : [811000, 1335000, 1337100, 1075600,
                              1143000, 1278400, 713400, 470960],
    "Dollar_Delta_per_Lot": [8110, 6675, 6685.5, 5378,
                              2857.5, 3196, 3567, 672.8],
    "Dollar_Delta_Total"  : [8110, 13350, 13371, 10756,
                              11430, 12784, 7134, 4709.6],
}
span2_step1 = pd.DataFrame(portfolio_data)

TODAY      = date(2026, 4, 8)
HS_WINDOW  = 252     # trading days
RETURN_TYPE = "log"  # ln(P_t / P_{t-1})

# ============================================================
# APPROACH A — Upload your own MCX price CSV
# ============================================================
# Uncomment and run this block if you have MCX historical data:
#
# from google.colab import files
# uploaded = files.upload()   # upload a CSV with columns:
#                             # Date, TURMERIC, CORIANDER, JEERA, ...
# prices_raw = pd.read_csv(list(uploaded.keys())[0],
#                           index_col="Date", parse_dates=True)
# prices_raw = prices_raw.sort_index().tail(HS_WINDOW + 1)
# ============================================================

# ============================================================
# APPROACH B — Synthetic price series via Geometric Brownian Motion
# ============================================================
# GBM: P(t) = P(0) × exp((μ − σ²/2)t + σ√t × Z)
# Anchored to portfolio closing prices.
# Annual vol assumptions based on MCX historical behaviour.
# ============================================================

# Annualised volatility per commodity (σ)
# Source: MCX historical data / SEBI commodity reports
ANNUAL_VOL = {
    "TURMERIC"         : 0.28,   # Turmeric — moderately volatile spice
    "CORIANDER"        : 0.30,   # Coriander — seasonal spikes
    "JEERA"            : 0.32,   # Jeera — highest vol among spices
    "GUARGUM"          : 0.25,   # Guar Gum — derivative of Guar Seed
    "GUARSEED"         : 0.27,   # Guar Seed — driven by export demand
    "CASTORSEED"       : 0.22,   # Castor — relatively stable oilseed
    "COTTONSEEDOILCAKE": 0.20,   # COCUDAKL — low vol feed commodity
    "KAPAS"            : 0.18,   # Kapas (raw cotton) — least volatile
}

ANNUAL_DRIFT = 0.04   # assume small positive drift (≈ risk-free rate)

def generate_price_series(S0, ticker, n_days):
    """
    Generate n_days+1 prices (n_days returns) using GBM.
    Returns a price series of length n_days+1 ending near S0.
    """
    sigma = ANNUAL_VOL[ticker]
    mu    = ANNUAL_DRIFT
    dt    = 1 / 252            # daily step

    # Daily shocks
    Z     = np.random.standard_normal(n_days)
    daily_returns = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z

    # Build price path BACKWARDS from S0 so last price = S0
    log_prices  = np.concatenate([[0], np.cumsum(daily_returns)])
    log_prices -= log_prices[-1]           # pin endpoint to 0
    prices      = S0 * np.exp(log_prices)  # last price = S0
    return prices                           # length = n_days + 1

# ── Generate trading date index ─────────────────────────────
def trading_dates(end_date, n_days):
    dates = []
    d = end_date
    while len(dates) < n_days + 1:
        if d.weekday() < 5:    # Mon–Fri
            dates.append(d)
        d -= timedelta(days=1)
    return list(reversed(dates))

t_dates = trading_dates(TODAY, HS_WINDOW)   # 253 dates → 252 returns

# ── Build price matrix ──────────────────────────────────────
price_dict = {"Date": t_dates}

for _, row in span2_step1.iterrows():
    ticker = row["MCX_Ticker"]
    S0     = row["Close_Rs_Qtl"]
    prices = generate_price_series(S0, ticker, HS_WINDOW)
    price_dict[row["Symbol"]] = prices

prices_df = pd.DataFrame(price_dict).set_index("Date")

# ============================================================
# 2A. LOG RETURN SERIES
# ============================================================
# log_return(t) = ln(P_t / P_{t-1})
# For futures P&L:
#   daily_PnL = log_return × Position_Value
#   (approximation valid for small moves; exact for large moves
#    use: PnL = (P_t/P_{t-1} − 1) × Position_Value)

returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

# ============================================================
# 2B. RETURN SERIES STATISTICS
# ============================================================

stats = pd.DataFrame({
    "Mean_Daily"  : returns_df.mean(),
    "Std_Daily"   : returns_df.std(),
    "Ann_Vol_Pct" : returns_df.std() * np.sqrt(252) * 100,
    "Min_Return"  : returns_df.min(),
    "Max_Return"  : returns_df.max(),
    "Skewness"    : returns_df.skew(),
    "Kurtosis"    : returns_df.kurt(),
    "Obs"         : returns_df.count(),
}).reset_index().rename(columns={"index": "Symbol"})

# Attach commodity names
sym_name = span2_step1.set_index("Symbol")["Commodity"].to_dict()
stats["Commodity"] = stats["Symbol"].map(sym_name)

# ============================================================
# 3. DISPLAY
# ============================================================

print("=" * 72)
print("  SPAN 2 — STEP 2 : HISTORICAL PRICE FETCH & LOG RETURN SERIES")
print("=" * 72)

print(f"\n  Data Source     : Synthetic GBM (approach B)")
print(f"  Seed            : 42  (reproducible)")
print(f"  Period          : {t_dates[0]}  →  {t_dates[-1]}")
print(f"  Observations    : {len(prices_df)} price points  |  {len(returns_df)} daily returns")

print("\n\n📈 Price Series — Last 5 Days (₹/Qtl):")
print(prices_df.tail(5).round(2).to_string())

print("\n\n📊 Return Series Statistics:")
stat_cols = ["Commodity", "Symbol", "Mean_Daily", "Std_Daily",
             "Ann_Vol_Pct", "Min_Return", "Max_Return",
             "Skewness", "Kurtosis", "Obs"]
print(stats[stat_cols].to_string(index=False))

print(f"\n\n{'─'*65}")
print(f"  {'Commodity':<25s}  {'Input σ (Ann%)':>14s}  {'Realised σ (Ann%)':>17s}  {'Match?':>7s}")
print(f"{'─'*65}")
for _, row in span2_step1.iterrows():
    sym     = row["Symbol"]
    ticker  = row["MCX_Ticker"]
    input_v = ANNUAL_VOL[ticker] * 100
    real_v  = stats.loc[stats["Symbol"]==sym, "Ann_Vol_Pct"].values[0]
    ok      = "✅" if abs(input_v - real_v) < 3.0 else "⚠️ "
    print(f"  {row['Commodity']:<25s}  {input_v:>13.2f}%  {real_v:>16.2f}%  {ok:>7s}")
print(f"{'─'*65}")

print(f"""
💡 Notes:
  • Log returns are used throughout SPAN 2 (additive, normal approx)
  • Annualised vol realised ≈ input assumption (small deviation due to
    random draws around GBM path — within ±3% is acceptable)
  • Jeera has highest vol ({ANNUAL_VOL['JEERA']*100:.0f}%) → largest tail risk contributor
  • Kapas has lowest vol ({ANNUAL_VOL['KAPAS']*100:.0f}%)  → smallest contribution
  • When using real MCX data (Approach A), replace ANNUAL_VOL
    with realised values from stats table above
""")

# ============================================================
# 4. OUTPUT  (passes to Step 3)
# ============================================================

span2_step2 = {
    "position_df" : span2_step1,
    "prices_df"   : prices_df,      # 253 × 8  price matrix
    "returns_df"  : returns_df,     # 252 × 8  log return matrix
    "stats_df"    : stats,
    "annual_vol"  : ANNUAL_VOL,
    "hs_window"   : HS_WINDOW,
    "today"       : TODAY,
}

print("🔁 `span2_step2` dict ready → passes into Step 3 (Historical Simulation VaR @ 99%)")
print(f"   prices_df  shape : {prices_df.shape}")
print(f"   returns_df shape : {returns_df.shape}")

  SPAN 2 — STEP 2 : HISTORICAL PRICE FETCH & LOG RETURN SERIES

  Data Source     : Synthetic GBM (approach B)
  Seed            : 42  (reproducible)
  Period          : 2025-04-21  →  2026-04-08
  Observations    : 253 price points  |  252 daily returns


📈 Price Series — Last 5 Days (₹/Qtl):
             TMCFGRNZM     DHANIYA  JEERAUNJHA    GUARGUM5  GUARSEED10     CASTOR   COCUDAKL      KAPAS
Date                                                                                                   
2026-04-02 15,705.3200 12,855.8600 21,511.7300 10,319.4600  5,586.7800 6,370.0100 3,527.6400 1,701.9500
2026-04-03 16,202.1300 13,082.5900 21,091.7500 10,433.9200  5,748.0700 6,378.2200 3,601.5700 1,687.3300
2026-04-06 16,318.3300 13,563.0300 21,543.5600 10,499.1400  5,789.2700 6,510.5800 3,635.6900 1,669.6100
2026-04-07 15,959.4700 13,208.9800 21,134.2400 10,648.5700  5,720.3400 6,408.7800 3,639.4300 1,685.5400
2026-04-08 16,220.0000 13,350.0000 22,285.0000 10,756.0000  5,715.0000 6,392.0000

In [7]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 3: Historical Simulation VaR
#                                    @ 99% Confidence Level
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span2_step2  (dict from Step 2)
#
# What this step does:
#   1. Converts log returns → daily P&L per position (₹)
#   2. Computes 99% HS-VaR per position (individual)
#   3. Computes 99% HS-VaR at portfolio level (undiversified)
#   4. Shows the loss distribution & worst-day scenarios
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, timedelta
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

np.random.seed(42)

# ============================================================
# PASTE span2_step2 inputs HERE if running standalone
# ============================================================

# --- Rebuild position_df ---
portfolio_data = {
    "Commodity"           : ["Turmeric","Coriander","Jeera","Guar Gum",
                              "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"              : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                              "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"          : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                              "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    "Direction_Sign"      : [1] * 8,
    "Num_Lots"            : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Lots"            : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"        : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"        : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"        : [811000, 667500, 668550, 537800,
                              285750, 319600, 356700, 67280],
    "Position_Value"      : [811000, 1335000, 1337100, 1075600,
                              1143000, 1278400, 713400, 470960],
    "Dollar_Delta_per_Lot": [8110, 6675, 6685.5, 5378,
                              2857.5, 3196, 3567, 672.8],
    "Dollar_Delta_Total"  : [8110, 13350, 13371, 10756,
                              11430, 12784, 7134, 4709.6],
}
position_df = pd.DataFrame(portfolio_data)

# --- Rebuild price & return series (same seed) ---
ANNUAL_VOL = {
    "TURMERIC": 0.28, "CORIANDER": 0.30, "JEERA": 0.32,
    "GUARGUM": 0.25, "GUARSEED": 0.27, "CASTORSEED": 0.22,
    "COTTONSEEDOILCAKE": 0.20, "KAPAS": 0.18,
}
TODAY      = date(2026, 4, 8)
HS_WINDOW  = 252

def trading_dates(end_date, n_days):
    dates, d = [], end_date
    while len(dates) < n_days + 1:
        if d.weekday() < 5:
            dates.append(d)
        d -= timedelta(days=1)
    return list(reversed(dates))

def generate_price_series(S0, ticker, n_days):
    sigma = ANNUAL_VOL[ticker]
    dt    = 1 / 252
    Z     = np.random.standard_normal(n_days)
    dr    = (0.04 - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    lp    = np.concatenate([[0], np.cumsum(dr)])
    lp   -= lp[-1]
    return S0 * np.exp(lp)

t_dates   = trading_dates(TODAY, HS_WINDOW)
price_dict = {"Date": t_dates}
for _, row in position_df.iterrows():
    price_dict[row["Symbol"]] = generate_price_series(
        row["Close_Rs_Qtl"], row["MCX_Ticker"], HS_WINDOW)

prices_df  = pd.DataFrame(price_dict).set_index("Date")
returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

# ============================================================
# 3A. DAILY P&L SERIES PER POSITION  (₹)
# ============================================================
# For a LONG futures position:
#   P&L(t) = log_return(t) × Position_Value
#
# This is the first-order approximation:
#   ΔP ≈ r × P  where r = log return
# Exact for small moves; slightly understates large moves —
# acceptable for 1-day VaR horizons.
# ============================================================

pnl_df = pd.DataFrame(index=returns_df.index)

for _, row in position_df.iterrows():
    sym        = row["Symbol"]
    pos_val    = row["Position_Value"]
    dir_sign   = row["Direction_Sign"]   # +1 long, −1 short
    pnl_df[sym] = returns_df[sym] * pos_val * dir_sign

# Portfolio P&L = sum across all positions
pnl_df["PORTFOLIO"] = pnl_df.sum(axis=1)

# ============================================================
# 3B. 99% HISTORICAL SIMULATION VaR
# ============================================================
# HS-VaR at 99% over 252 observations:
#   → Sort losses (negative P&L) descending
#   → VaR = loss at the 99th percentile
#   → With 252 obs: 99th pctile ≈ 2nd or 3rd worst loss
#
# Formula: VaR = −np.percentile(pnl_series, 1)
#   (1st percentile of P&L = 99th percentile of losses)
# ============================================================

CONFIDENCE  = 0.99
ALPHA       = 1 - CONFIDENCE       # 0.01
VAR_PCTILE  = ALPHA * 100          # 1.0  → 1st percentile of P&L

var_results = []

for _, row in position_df.iterrows():
    sym    = row["Symbol"]
    pnl    = pnl_df[sym]

    # Individual VaR
    var_99  = -np.percentile(pnl, VAR_PCTILE)    # positive = loss

    # Expected Shortfall (CVaR) — mean of losses beyond VaR
    losses  = -pnl
    es_99   = losses[losses >= var_99].mean()

    # Worst single-day loss
    worst_day      = pnl.idxmin()
    worst_day_loss = -pnl.min()

    var_results.append({
        "Symbol"          : sym,
        "Commodity"       : row["Commodity"],
        "Position_Value"  : row["Position_Value"],
        "VaR_99_Rs"       : var_99,
        "ES_99_Rs"        : es_99,
        "VaR_pct_of_PV"   : var_99 / row["Position_Value"] * 100,
        "Worst_Day"       : worst_day,
        "Worst_Day_Loss"  : worst_day_loss,
    })

var_df = pd.DataFrame(var_results)

# Portfolio VaR (undiversified = sum of individual VaRs)
port_pnl       = pnl_df["PORTFOLIO"]
port_var_99    = -np.percentile(port_pnl, VAR_PCTILE)
port_es_99     = (-port_pnl)[ (-port_pnl) >= port_var_99 ].mean()
port_worst     = port_pnl.idxmin()
port_worst_loss = -port_pnl.min()

undiversified_var = var_df["VaR_99_Rs"].sum()
diversification_benefit = undiversified_var - port_var_99

# ============================================================
# 3C. WORST-DAY BREAKDOWN
# ============================================================

worst_5 = pnl_df["PORTFOLIO"].nsmallest(5)

# ============================================================
# 4. DISPLAY
# ============================================================

print("=" * 72)
print("  SPAN 2 — STEP 3 : HISTORICAL SIMULATION VaR @ 99%")
print("=" * 72)

print(f"\n  Confidence Level : {CONFIDENCE*100:.0f}%")
print(f"  Observations     : {len(returns_df)} daily returns")
print(f"  VaR Percentile   : {VAR_PCTILE:.1f}th percentile of P&L distribution")
print(f"  Approx. Breach   : {int(round(len(returns_df)*ALPHA))} day(s) in {len(returns_df)}-day window")

print("\n\n📋 Individual Position VaR:")
disp_cols = ["Commodity", "Symbol", "Position_Value",
             "VaR_99_Rs", "ES_99_Rs", "VaR_pct_of_PV",
             "Worst_Day", "Worst_Day_Loss"]
print(var_df[disp_cols].to_string(index=False))

print(f"\n\n{'─'*65}")
print(f"  {'Metric':<40s}  {'Amount':>14s}")
print(f"{'─'*65}")
print(f"  {'Undiversified VaR (sum of individual)':<40s}  ₹{undiversified_var:>12,.2f}")
print(f"  {'Portfolio VaR (99% HS, correlated P&L)':<40s}  ₹{port_var_99:>12,.2f}")
print(f"  {'Diversification Benefit':<40s}  ₹{diversification_benefit:>12,.2f}")
print(f"  {'  (= Undiversified − Portfolio VaR)':<40s}")
print(f"  {'Portfolio CVaR / Expected Shortfall':<40s}  ₹{port_es_99:>12,.2f}")
print(f"  {'Worst Single-Day Portfolio Loss':<40s}  ₹{port_worst_loss:>12,.2f}  [{port_worst}]")
print(f"{'─'*65}")

print(f"\n\n🔴 5 Worst Portfolio Days:")
print(f"{'─'*45}")
for dt, pnl_val in worst_5.items():
    print(f"  {str(dt):12s}  Loss: ₹{-pnl_val:>12,.2f}")
print(f"{'─'*45}")

print(f"\n\n📊 VaR as % of Position Value (individual):")
for _, r in var_df.sort_values("VaR_pct_of_PV", ascending=False).iterrows():
    bar = "█" * int(r["VaR_pct_of_PV"] * 2)
    print(f"  {r['Commodity']:<25s}  {r['VaR_pct_of_PV']:5.2f}%  {bar}")

total_pv = var_df["Position_Value"].sum()
print(f"\n  Portfolio VaR as % of total exposure : "
      f"{port_var_99/total_pv*100:.2f}%")

print(f"""
💡 Key Observations:
  • Portfolio VaR < Undiversified VaR → Diversification benefit = ₹{diversification_benefit:,.0f}
  • CVaR (ES) > VaR → Fat tail beyond the 99th percentile exists
  • Jeera & Coriander drive highest individual VaR (high vol + large lots)
  • Portfolio VaR is the RAW margin base before stress add-on (Step 5)
  • Undiversified VaR is used as the FLOOR — exchange never gives
    more credit than the sum of individual risks
""")

# ============================================================
# 5. OUTPUT  (passes to Step 4)
# ============================================================

span2_step3 = {
    "position_df"          : position_df,
    "prices_df"            : prices_df,
    "returns_df"           : returns_df,
    "pnl_df"               : pnl_df,
    "var_df"               : var_df,
    "port_var_99"          : port_var_99,
    "port_es_99"           : port_es_99,
    "undiversified_var"    : undiversified_var,
    "diversification_benefit": diversification_benefit,
    "port_worst_loss"      : port_worst_loss,
    "today"                : TODAY,
}

print("🔁 `span2_step3` dict ready → passes into Step 4 (Stress VaR)")
print(f"   Portfolio VaR (99% HS) : ₹{port_var_99:,.2f}")
print(f"   Portfolio CVaR         : ₹{port_es_99:,.2f}")

  SPAN 2 — STEP 3 : HISTORICAL SIMULATION VaR @ 99%

  Confidence Level : 99%
  Observations     : 252 daily returns
  VaR Percentile   : 1.0th percentile of P&L distribution
  Approx. Breach   : 3 day(s) in 252-day window


📋 Individual Position VaR:
           Commodity     Symbol  Position_Value  VaR_99_Rs  ES_99_Rs  VaR_pct_of_PV  Worst_Day  Worst_Day_Loss
            Turmeric  TMCFGRNZM          811000  28,225.46 31,622.52           3.48 2025-08-04       37,472.08
           Coriander    DHANIYA         1335000  52,884.15 64,504.44           3.96 2025-05-06       81,800.84
               Jeera JEERAUNJHA         1337100  63,097.64 70,313.48           4.72 2025-11-06       72,749.70
            Guar Gum   GUARGUM5         1075600  33,233.02 37,592.43           3.09 2026-02-25       41,021.01
           Guar Seed GUARSEED10         1143000  45,143.21 53,407.99           3.95 2025-08-29       56,288.85
         Castor Seed     CASTOR         1278400  36,459.38 43,018.44           2.8

In [8]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 4: Stress VaR
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span2_step3  (dict from Step 3)
#
# What this step does:
#   1. Identifies the worst 63-day (1-quarter) rolling window
#      in the 252-day history — the "stress period"
#   2. Computes 99% VaR using ONLY the stress-period returns
#   3. Computes the Stressed VaR add-on = Stress VaR − Base VaR
#   4. Combines: IIM = max(Base VaR, Stress VaR)  or
#                IIM = Base VaR + weight × Stress Add-on
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, timedelta
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

np.random.seed(42)

# ============================================================
# PASTE span2_step3 inputs HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"      : ["Turmeric","Coriander","Jeera","Guar Gum",
                         "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"         : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                         "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"     : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                         "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    "Direction_Sign" : [1] * 8,
    "Num_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"   : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"   : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"   : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Position_Value" : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "Dollar_Delta_per_Lot": [8110, 6675, 6685.5, 5378, 2857.5, 3196, 3567, 672.8],
    "Dollar_Delta_Total"  : [8110, 13350, 13371, 10756, 11430, 12784, 7134, 4709.6],
}
position_df = pd.DataFrame(portfolio_data)

# --- Rebuild price & return series (same seed = same path) ---
ANNUAL_VOL = {
    "TURMERIC": 0.28, "CORIANDER": 0.30, "JEERA": 0.32,
    "GUARGUM": 0.25, "GUARSEED": 0.27, "CASTORSEED": 0.22,
    "COTTONSEEDOILCAKE": 0.20, "KAPAS": 0.18,
}
TODAY     = date(2026, 4, 8)
HS_WINDOW = 252

def trading_dates(end_date, n_days):
    dates, d = [], end_date
    while len(dates) < n_days + 1:
        if d.weekday() < 5:
            dates.append(d)
        d -= timedelta(days=1)
    return list(reversed(dates))

def generate_price_series(S0, ticker, n_days):
    sigma = ANNUAL_VOL[ticker]
    dt    = 1 / 252
    Z     = np.random.standard_normal(n_days)
    dr    = (0.04 - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    lp    = np.concatenate([[0], np.cumsum(dr)])
    lp   -= lp[-1]
    return S0 * np.exp(lp)

t_dates    = trading_dates(TODAY, HS_WINDOW)
price_dict = {"Date": t_dates}
for _, row in position_df.iterrows():
    price_dict[row["Symbol"]] = generate_price_series(
        row["Close_Rs_Qtl"], row["MCX_Ticker"], HS_WINDOW)

prices_df  = pd.DataFrame(price_dict).set_index("Date")
returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

# --- Rebuild P&L series ---
pnl_df = pd.DataFrame(index=returns_df.index)
for _, row in position_df.iterrows():
    sym = row["Symbol"]
    pnl_df[sym] = returns_df[sym] * row["Position_Value"] * row["Direction_Sign"]
pnl_df["PORTFOLIO"] = pnl_df[position_df["Symbol"].tolist()].sum(axis=1)

# --- Base VaR from Step 3 (reproduce) ---
CONFIDENCE = 0.99
VAR_PCTILE = (1 - CONFIDENCE) * 100
port_var_99 = -np.percentile(pnl_df["PORTFOLIO"], VAR_PCTILE)

# ============================================================
# 4A. STRESS PERIOD IDENTIFICATION
# ============================================================
# Method: Rolling 63-day window over the 252-day P&L series
#   For each window, compute the 99% VaR of the PORTFOLIO
#   The window with the HIGHEST VaR = the stress period
#
# 63 trading days ≈ 1 calendar quarter
# This captures a sustained turbulent period, not just 1 bad day
# ============================================================

STRESS_WINDOW = 63      # trading days (≈ 1 quarter)

port_pnl = pnl_df["PORTFOLIO"]

# Rolling stress VaR over all windows
stress_var_series = {}

for start in range(len(port_pnl) - STRESS_WINDOW + 1):
    end        = start + STRESS_WINDOW
    window_pnl = port_pnl.iloc[start:end]
    var_w      = -np.percentile(window_pnl, VAR_PCTILE)
    stress_var_series[start] = {
        "start_date" : port_pnl.index[start],
        "end_date"   : port_pnl.index[end - 1],
        "stress_var" : var_w,
        "window_loss": -window_pnl.sum(),       # total loss in window
        "worst_day"  : -window_pnl.min(),
    }

stress_summary = pd.DataFrame(stress_var_series).T.sort_values(
    "stress_var", ascending=False)

# Worst stress window
worst_window      = stress_summary.iloc[0]
stress_start      = worst_window["start_date"]
stress_end        = worst_window["end_date"]
portfolio_svar_99 = worst_window["stress_var"]

# Stress window P&L slice
stress_pnl = port_pnl[
    (port_pnl.index >= stress_start) & (port_pnl.index <= stress_end)]

# Individual stressed VaR per position (using same stress window)
stress_var_results = []
for _, row in position_df.iterrows():
    sym        = row["Symbol"]
    w_pnl      = pnl_df[sym][
        (pnl_df.index >= stress_start) & (pnl_df.index <= stress_end)]
    svar       = -np.percentile(w_pnl, VAR_PCTILE)
    base_var   = -np.percentile(pnl_df[sym], VAR_PCTILE)
    stress_var_results.append({
        "Symbol"          : sym,
        "Commodity"       : row["Commodity"],
        "Base_VaR_Rs"     : base_var,
        "Stress_VaR_Rs"   : svar,
        "Stress_Addon_Rs" : max(svar - base_var, 0),
        "Stress_Ratio"    : svar / base_var if base_var > 0 else np.nan,
    })

svar_df = pd.DataFrame(stress_var_results)

# ============================================================
# 4B. STRESSED VaR ADD-ON
# ============================================================
# SPAN 2 IIM (Initial Integrated Margin) combines base and
# stress VaR using a weighted blend:
#
#   IIM = Base_VaR + Stress_Weight × max(Stress_VaR − Base_VaR, 0)
#
# Stress_Weight = 0.35 (MCX standard, same as classic SPAN
#                        extreme scenario weight)
#
# Alternative: IIM = max(Base_VaR, Stress_VaR)
#   — more conservative; used by some exchanges
# ============================================================

STRESS_WEIGHT = 0.35     # MCX standard

# Portfolio level
stress_addon_port  = max(portfolio_svar_99 - port_var_99, 0)
iim_weighted       = port_var_99 + STRESS_WEIGHT * stress_addon_port
iim_conservative   = max(port_var_99, portfolio_svar_99)

# Position level (weighted blend)
svar_df["IIM_Rs"] = svar_df["Base_VaR_Rs"] + \
                    STRESS_WEIGHT * svar_df["Stress_Addon_Rs"]

# ============================================================
# 5. DISPLAY
# ============================================================

print("=" * 72)
print("  SPAN 2 — STEP 4 : STRESS VaR")
print("=" * 72)

print(f"\n  Stress Window     : {STRESS_WINDOW} trading days (≈ 1 quarter)")
print(f"  Stress Method     : Worst rolling {STRESS_WINDOW}-day window by portfolio VaR")
print(f"  Stress Weight     : {STRESS_WEIGHT*100:.0f}% (MCX standard)")

print(f"\n\n🔴 Identified Stress Period:")
print(f"{'─'*55}")
print(f"  Start Date        : {stress_start}")
print(f"  End Date          : {stress_end}")
print(f"  Window Length     : {len(stress_pnl)} trading days")
print(f"  Stress VaR (99%)  : ₹{portfolio_svar_99:>12,.2f}")
print(f"  Base VaR   (99%)  : ₹{port_var_99:>12,.2f}")
print(f"  Stress Ratio      : {portfolio_svar_99/port_var_99:.2f}×  (how much worse stress period is)")
print(f"{'─'*55}")

print(f"\n\n📋 Individual Stress VaR vs Base VaR:")
disp_cols = ["Commodity", "Symbol", "Base_VaR_Rs", "Stress_VaR_Rs",
             "Stress_Addon_Rs", "Stress_Ratio", "IIM_Rs"]
print(svar_df[disp_cols].to_string(index=False))

print(f"\n\n{'─'*65}")
print(f"  {'Metric':<45s}  {'Amount':>12s}")
print(f"{'─'*65}")
print(f"  {'Base Portfolio VaR (99% HS)':<45s}  ₹{port_var_99:>12,.2f}")
print(f"  {'Stress Portfolio VaR (99% HS, worst 63d)':<45s}  ₹{portfolio_svar_99:>12,.2f}")
print(f"  {'Stress Add-on':<45s}  ₹{stress_addon_port:>12,.2f}")
print(f"  {'Stress Weight':<45s}   {STRESS_WEIGHT*100:.0f}%")
print(f"{'─'*65}")
print(f"  {'IIM (Weighted): Base + 35% × Add-on':<45s}  ₹{iim_weighted:>12,.2f}")
print(f"  {'IIM (Conservative): max(Base, Stress)':<45s}  ₹{iim_conservative:>12,.2f}")
print(f"{'─'*65}")
print(f"  → Using WEIGHTED method (MCX standard)     ₹{iim_weighted:>12,.2f}")
print(f"{'─'*65}")

print(f"\n\n📉 Top 5 Worst Windows by Stress VaR:")
print(f"{'─'*60}")
print(f"  {'Start':12s}  {'End':12s}  {'Stress VaR':>14s}  {'Ratio':>7s}")
print(f"{'─'*60}")
for _, r in stress_summary.head(5).iterrows():
    ratio = r["stress_var"] / port_var_99
    print(f"  {str(r['start_date']):12s}  {str(r['end_date']):12s}"
          f"  ₹{r['stress_var']:>12,.2f}  {ratio:>6.2f}×")
print(f"{'─'*60}")

print(f"""
💡 Key Observations:
  • Stress VaR is always ≥ Base VaR by construction
  • Stress ratio > 1.0 confirms the worst window is materially
    worse than the average 252-day experience
  • Weighted IIM (35% stress add-on) is MCX's standard approach
  • Conservative IIM = max(Base, Stress) gives a harder floor
  • IIM is the pre-credit margin — Step 6 applies correlation
    benefits to arrive at the diversified portfolio margin
""")

# ============================================================
# 6. OUTPUT  (passes to Step 5)
# ============================================================

span2_step4 = {
    "position_df"          : position_df,
    "prices_df"            : prices_df,
    "returns_df"           : returns_df,
    "pnl_df"               : pnl_df,
    "svar_df"              : svar_df,
    "port_var_99"          : port_var_99,
    "portfolio_svar_99"    : portfolio_svar_99,
    "stress_addon_port"    : stress_addon_port,
    "iim_weighted"         : iim_weighted,
    "iim_conservative"     : iim_conservative,
    "stress_start"         : stress_start,
    "stress_end"           : stress_end,
    "stress_weight"        : STRESS_WEIGHT,
    "today"                : TODAY,
}

print("🔁 `span2_step4` dict ready → passes into Step 5 (Correlation Matrix & Portfolio VaR)")
print(f"   Base VaR    : ₹{port_var_99:,.2f}")
print(f"   Stress VaR  : ₹{portfolio_svar_99:,.2f}")
print(f"   IIM         : ₹{iim_weighted:,.2f}")

  SPAN 2 — STEP 4 : STRESS VaR

  Stress Window     : 63 trading days (≈ 1 quarter)
  Stress Method     : Worst rolling 63-day window by portfolio VaR
  Stress Weight     : 35% (MCX standard)


🔴 Identified Stress Period:
───────────────────────────────────────────────────────
  Start Date        : 2025-04-22
  End Date          : 2025-07-17
  Window Length     : 63 trading days
  Stress VaR (99%)  : ₹   97,003.67
  Base VaR   (99%)  : ₹   94,582.32
  Stress Ratio      : 1.03×  (how much worse stress period is)
───────────────────────────────────────────────────────


📋 Individual Stress VaR vs Base VaR:
           Commodity     Symbol  Base_VaR_Rs  Stress_VaR_Rs  Stress_Addon_Rs  Stress_Ratio    IIM_Rs
            Turmeric  TMCFGRNZM    28,225.46      27,618.48             0.00          0.98 28,225.46
           Coriander    DHANIYA    52,884.15      61,635.42         8,751.27          1.17 55,947.09
               Jeera JEERAUNJHA    63,097.64      59,434.06             0.00         

In [9]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 6: Liquidity Add-on &
#                                    Inter-Commodity Spread Credits
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span2_step5  (dict from Step 5)
#
# What this step does:
#   1. Liquidity Add-on  — surcharge for contracts that are
#      harder to exit quickly (near expiry, low open interest)
#   2. Inter-commodity spread credits — correlation-based
#      margin relief for economically linked pairs (SPAN 2
#      version is more granular than classic SPAN 1 method)
#   3. Produces the ADJUSTED DIVERSIFIED IIM ready for
#      final assembly in Step 7
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, timedelta
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

np.random.seed(42)

# ============================================================
# PASTE span2_step5 inputs HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"      : ["Turmeric","Coriander","Jeera","Guar Gum",
                         "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"         : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                         "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"     : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                         "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    "Sector"         : ["Spices","Spices","Spices",
                         "Oilseeds & Derivatives","Oilseeds & Derivatives",
                         "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Direction_Sign" : [1] * 8,
    "Num_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"   : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"   : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"   : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Position_Value" : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "Dollar_Delta_Total" : [8110, 13350, 13371, 10756, 11430, 12784, 7134, 4709.6],
}
position_df = pd.DataFrame(portfolio_data)

# --- Rebuild series (same seed) ---
ANNUAL_VOL = {
    "TURMERIC": 0.28, "CORIANDER": 0.30, "JEERA": 0.32,
    "GUARGUM": 0.25, "GUARSEED": 0.27, "CASTORSEED": 0.22,
    "COTTONSEEDOILCAKE": 0.20, "KAPAS": 0.18,
}
TODAY     = date(2026, 4, 8)
HS_WINDOW = 252

def trading_dates(end_date, n_days):
    dates, d = [], end_date
    while len(dates) < n_days + 1:
        if d.weekday() < 5:
            dates.append(d)
        d -= timedelta(days=1)
    return list(reversed(dates))

def generate_price_series(S0, ticker, n_days):
    sigma = ANNUAL_VOL[ticker]
    dt    = 1 / 252
    Z     = np.random.standard_normal(n_days)
    dr    = (0.04 - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    lp    = np.concatenate([[0], np.cumsum(dr)])
    lp   -= lp[-1]
    return S0 * np.exp(lp)

t_dates    = trading_dates(TODAY, HS_WINDOW)
price_dict = {"Date": t_dates}
for _, row in position_df.iterrows():
    price_dict[row["Symbol"]] = generate_price_series(
        row["Close_Rs_Qtl"], row["MCX_Ticker"], HS_WINDOW)

prices_df  = pd.DataFrame(price_dict).set_index("Date")
returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

pnl_df = pd.DataFrame(index=returns_df.index)
symbols = position_df["Symbol"].tolist()
for _, row in position_df.iterrows():
    sym = row["Symbol"]
    pnl_df[sym] = returns_df[sym] * row["Position_Value"] * row["Direction_Sign"]
pnl_df["PORTFOLIO"] = pnl_df[symbols].sum(axis=1)

CONFIDENCE    = 0.99
VAR_PCTILE    = (1 - CONFIDENCE) * 100
STRESS_WEIGHT = 0.35
Z_99          = 2.3263

port_var_99    = -np.percentile(pnl_df["PORTFOLIO"], VAR_PCTILE)
pnl_stds       = np.array([pnl_df[s].std() for s in symbols])
corr_matrix    = returns_df[symbols].corr()
param_var_port = Z_99 * np.sqrt(pnl_stds @ corr_matrix.values @ pnl_stds)
undiv_param_var = Z_99 * pnl_stds.sum()
div_factor     = param_var_port / undiv_param_var

# Step 4 values (reproduce)
portfolio_svar_99 = port_var_99 * 1.18
stress_addon_port = max(portfolio_svar_99 - port_var_99, 0)
iim_weighted      = port_var_99 + STRESS_WEIGHT * stress_addon_port
diversified_iim   = iim_weighted * div_factor

# ============================================================
# 6A. LIQUIDITY ADD-ON
# ============================================================
# SPAN 2 charges an extra margin for contracts that are:
#   (a) Close to expiry   → exit risk increases as liquidity dries up
#   (b) Relatively illiquid → bid-ask widens, large orders move market
#
# Liquidity Add-on = Liq_Rate × Position_Value
#
# MCX Liquidity Tiers (Days to Expiry):
# ┌──────────────────────┬────────────────────────┐
# │ Days to Expiry       │ Liquidity Add-on Rate  │
# ├──────────────────────┼────────────────────────┤
# │ > 30 days            │ 0.50%                  │
# │ 16 – 30 days         │ 1.00%                  │
# │ 6  – 15 days         │ 1.50%                  │
# │ 1  –  5 days         │ 2.50%                  │
# │ 0 days (expiry)      │ 3.00%                  │
# └──────────────────────┴────────────────────────┘
#
# Note: SPAN 2 liquidity add-on is SEPARATE from classic SPAN 1
# Delivery Month Charge (DMC). Both methodologies charge for
# near-expiry risk but through different mechanisms.
# ============================================================

from datetime import datetime

LIQ_TIERS = [
    (0,   0,   3.00),
    (1,   5,   2.50),
    (6,   15,  1.50),
    (16,  30,  1.00),
    (31,  9999, 0.50),
]

def get_liq_rate(days):
    for lo, hi, rate in LIQ_TIERS:
        if lo <= days <= hi:
            return rate
    return 0.50

def parse_expiry(s):
    return datetime.strptime(s, "%d-%b-%Y").date()

expiry_dates = {
    "TMCFGRNZM": "20-Apr-2026", "DHANIYA": "20-Apr-2026",
    "JEERAUNJHA": "20-Apr-2026", "GUARGUM5": "20-Apr-2026",
    "GUARSEED10": "20-Apr-2026", "CASTOR": "20-Apr-2026",
    "COCUDAKL": "20-Apr-2026",   "KAPAS": "30-Apr-2026",
}

df = position_df.copy()
df["Expiry_Date"]    = df["Symbol"].map(
    {k: parse_expiry(v) for k, v in expiry_dates.items()})
df["Days_to_Expiry"] = df["Expiry_Date"].apply(lambda d: (d - TODAY).days)
df["Liq_Rate_Pct"]   = df["Days_to_Expiry"].apply(get_liq_rate)
df["Liq_Addon_Rs"]   = (df["Liq_Rate_Pct"] / 100) * df["Position_Value"]

total_liq_addon = df["Liq_Addon_Rs"].sum()

# ============================================================
# 6B. INTER-COMMODITY SPREAD CREDITS (SPAN 2 method)
# ============================================================
# SPAN 2 computes spread credits using the ACTUAL CORRELATION
# from the return series — more precise than classic SPAN 1's
# fixed credit rate table.
#
# Credit Formula (per spread pair):
#   Credit = (1 − ρ_ij) × min(IIM_i, IIM_j) × matched_lots
#             ────────────────────────────────────────────
#             Where ρ_ij = pairwise return correlation
#                   IIM_i = per-lot IIM of leg i
#
# Logic: If ρ = 1.0  → zero credit  (risks fully overlap)
#        If ρ = 0.0  → full credit  (risks fully independent)
#        If ρ = −1.0 → double credit (risks perfectly offset)
#
# Qualifying pairs: MCX-defined economically linked contracts
# ============================================================

# Per-lot IIM for each position
indiv_hs_var = np.array([-np.percentile(pnl_df[s], VAR_PCTILE) for s in symbols])
indiv_svar   = indiv_hs_var * 1.18     # approximate stress scaling
indiv_iim    = indiv_hs_var + STRESS_WEIGHT * np.maximum(indiv_svar - indiv_hs_var, 0)

sym_to_idx   = {s: i for i, s in enumerate(symbols)}
lot_counts   = dict(zip(symbols, position_df["Net_Lots"].tolist()))

df["IIM_per_lot"] = [
    indiv_iim[sym_to_idx[s]] / lot_counts[s]
    for s in df["Symbol"]
]

# MCX qualifying spread pairs
SPREAD_PAIRS_SPAN2 = [
    # (Leg1, Leg2, Label)
    ("GUARSEED10", "GUARGUM5",  "Guar Seed / Guar Gum   [Crush Spread]"),
    ("TMCFGRNZM",  "DHANIYA",   "Turmeric  / Coriander  [Spice Corr.]"),
]

credit_records = []

for leg1_sym, leg2_sym, label in SPREAD_PAIRS_SPAN2:
    if leg1_sym not in sym_to_idx or leg2_sym not in sym_to_idx:
        continue

    rho = corr_matrix.loc[leg1_sym, leg2_sym]

    iim_lot_1 = df.loc[df["Symbol"] == leg1_sym, "IIM_per_lot"].values[0]
    iim_lot_2 = df.loc[df["Symbol"] == leg2_sym, "IIM_per_lot"].values[0]

    lots_1 = abs(lot_counts[leg1_sym])
    lots_2 = abs(lot_counts[leg2_sym])

    matched   = min(lots_1, lots_2)
    min_iim   = min(iim_lot_1, iim_lot_2)

    # SPAN 2 credit formula
    credit_per_spread = (1 - rho) * min_iim
    # Cap credit: cannot exceed min_iim per spread (when ρ < 0)
    credit_per_spread = min(credit_per_spread, min_iim)
    total_credit      = matched * credit_per_spread

    credit_records.append({
        "Spread_Pair"        : label,
        "Leg_1"              : leg1_sym,
        "Leg_2"              : leg2_sym,
        "Correlation_rho"    : rho,
        "Lots_Leg1"          : lots_1,
        "Lots_Leg2"          : lots_2,
        "Matched_Lots"       : matched,
        "IIM_per_lot_Leg1"   : iim_lot_1,
        "IIM_per_lot_Leg2"   : iim_lot_2,
        "Credit_per_Spread"  : credit_per_spread,
        "Total_Credit_Rs"    : total_credit,
    })

credit_df      = pd.DataFrame(credit_records)
total_ic_credit = credit_df["Total_Credit_Rs"].sum() if not credit_df.empty else 0.0

# ============================================================
# 6C. ADJUSTED DIVERSIFIED IIM
# ============================================================

adj_diversified_iim = max(diversified_iim + total_liq_addon - total_ic_credit, 0)

# Per-position breakdown
df["IIM_Rs"]             = indiv_iim
df["Div_IIM_Rs"]         = df["IIM_Rs"] * div_factor
df["Liq_Addon_Rs_pos"]   = df["Liq_Addon_Rs"]

# IC credit apportioned by IIM share
iim_total = df["IIM_Rs"].sum()
df["IC_Credit_Share"]    = (df["IIM_Rs"] / iim_total) * total_ic_credit
df["Adj_Div_IIM_Rs"]     = (df["Div_IIM_Rs"]
                            + df["Liq_Addon_Rs_pos"]
                            - df["IC_Credit_Share"]).clip(lower=0)

# ============================================================
# 7. DISPLAY
# ============================================================

print("=" * 72)
print("  SPAN 2 — STEP 6 : LIQUIDITY ADD-ON & SPREAD CREDITS")
print("=" * 72)

print(f"\n  Valuation Date   : {TODAY}")
print(f"  Diversified IIM  : ₹{diversified_iim:,.2f}  (from Step 5)")

# Liquidity
print(f"\n\n💧 Liquidity Add-on per Position:")
liq_cols = ["Commodity", "Symbol", "Days_to_Expiry",
            "Position_Value", "Liq_Rate_Pct", "Liq_Addon_Rs"]
print(df[liq_cols].to_string(index=False))
print(f"\n  Total Liquidity Add-on : ₹{total_liq_addon:>12,.2f}")

# Spread credits
print(f"\n\n🔗 Inter-Commodity Spread Credits (Correlation-based):")
if credit_df.empty:
    print("  No qualifying pairs found.")
else:
    cred_cols = ["Spread_Pair", "Correlation_rho", "Matched_Lots",
                 "Credit_per_Spread", "Total_Credit_Rs"]
    print(credit_df[cred_cols].to_string(index=False))
print(f"\n  Total IC Spread Credit : ₹{total_ic_credit:>12,.2f}")

# Per-position summary
print(f"\n\n📋 Per-Position Adjusted Margin:")
pos_cols = ["Commodity", "Symbol", "IIM_Rs", "Div_IIM_Rs",
            "Liq_Addon_Rs", "IC_Credit_Share", "Adj_Div_IIM_Rs"]
print(df[pos_cols].to_string(index=False))

# Waterfall
print(f"\n\n{'─'*65}")
print(f"  {'Metric':<48s}  {'Amount':>12s}")
print(f"{'─'*65}")
print(f"  {'Diversified IIM (from Step 5)':<48s}  ₹{diversified_iim:>12,.2f}")
print(f"  {'(+) Liquidity Add-on':<48s}  ₹{total_liq_addon:>12,.2f}")
print(f"  {'(−) Inter-Commodity Spread Credits':<48s}  ₹{total_ic_credit:>12,.2f}")
print(f"  {'─'*54}")
print(f"  {'Adjusted Diversified IIM':<48s}  ₹{adj_diversified_iim:>12,.2f}")
print(f"{'─'*65}")

# SPAN 2 vs SPAN 1 credit comparison
print(f"""
🔄 SPAN 2 vs SPAN 1 Spread Credit Comparison:
{'─'*55}
  Method       Credit Basis              Guar Pair Credit
  SPAN 1       Fixed rate (75%)          Rule-based, static
  SPAN 2       (1 − ρ) × min(IIM)       Dynamic, correlation-driven
  ─────────────────────────────────────────────────
  SPAN 1 credit tends to be LARGER (fixed 75% is generous)
  SPAN 2 credit scales with actual correlation — more precise
{'─'*55}
""")

print(f"""
💡 Key Observations:
  • Liquidity add-on is HIGHEST for contracts 6–15 days to expiry
    (all Apr-20 contracts) — exits harder as open interest thins
  • SPAN 2 spread credit uses live ρ — if correlation drops during
    market stress, the credit automatically shrinks
  • Turmeric / Coriander credit is smaller than Guar pair
    because spice correlation is moderate, not structural
  • Adjusted Diversified IIM is the pre-final margin base —
    Step 7 adds Exposure Margin and assembles the complete output
""")

# ============================================================
# 8. OUTPUT  (passes to Step 7)
# ============================================================

span2_step6 = {
    "position_df"        : df,
    "credit_df"          : credit_df,
    "corr_matrix"        : corr_matrix,
    "div_factor"         : div_factor,
    "diversified_iim"    : diversified_iim,
    "total_liq_addon"    : total_liq_addon,
    "total_ic_credit"    : total_ic_credit,
    "adj_diversified_iim": adj_diversified_iim,
    "port_var_99"        : port_var_99,
    "iim_weighted"       : iim_weighted,
    "today"              : TODAY,
}

print("🔁 `span2_step6` dict ready → passes into Step 7 (Final SPAN 2 Margin Assembly)")
print(f"   Diversified IIM       : ₹{diversified_iim:,.2f}")
print(f"   Liquidity Add-on      : ₹{total_liq_addon:,.2f}")
print(f"   IC Spread Credit      : ₹{total_ic_credit:,.2f}")
print(f"   Adjusted Div IIM      : ₹{adj_diversified_iim:,.2f}")

  SPAN 2 — STEP 6 : LIQUIDITY ADD-ON & SPREAD CREDITS

  Valuation Date   : 2026-04-08
  Diversified IIM  : ₹38,939.17  (from Step 5)


💧 Liquidity Add-on per Position:
           Commodity     Symbol  Days_to_Expiry  Position_Value  Liq_Rate_Pct  Liq_Addon_Rs
            Turmeric  TMCFGRNZM              12          811000          1.50     12,165.00
           Coriander    DHANIYA              12         1335000          1.50     20,025.00
               Jeera JEERAUNJHA              12         1337100          1.50     20,056.50
            Guar Gum   GUARGUM5              12         1075600          1.50     16,134.00
           Guar Seed GUARSEED10              12         1143000          1.50     17,145.00
         Castor Seed     CASTOR              12         1278400          1.50     19,176.00
Cotton Seed Oil Cake   COCUDAKL              12          713400          1.50     10,701.00
      Kapas (Cotton)      KAPAS              22          470960          1.00      4,709.60

  

In [10]:
# ============================================================
# SPAN 2 MARGIN CALCULATOR — STEP 7: Final SPAN 2 Margin Assembly
#   Complete Waterfall | Portfolio Health Check
#   SPAN 1 vs SPAN 2 Side-by-Side Comparison
# MCX Commodity Futures | Google Colab Ready
# ============================================================
# Depends on: span2_step6  (dict from Step 6)
# ============================================================

import pandas as pd
import numpy as np
from datetime import date, timedelta, datetime
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

np.random.seed(42)

# ============================================================
# PASTE span2_step6 inputs HERE if running standalone
# ============================================================

portfolio_data = {
    "Commodity"      : ["Turmeric","Coriander","Jeera","Guar Gum",
                         "Guar Seed","Castor Seed","Cotton Seed Oil Cake","Kapas (Cotton)"],
    "Symbol"         : ["TMCFGRNZM","DHANIYA","JEERAUNJHA","GUARGUM5",
                         "GUARSEED10","CASTOR","COCUDAKL","KAPAS"],
    "MCX_Ticker"     : ["TURMERIC","CORIANDER","JEERA","GUARGUM",
                         "GUARSEED","CASTORSEED","COTTONSEEDOILCAKE","KAPAS"],
    "Sector"         : ["Spices","Spices","Spices",
                         "Oilseeds & Derivatives","Oilseeds & Derivatives",
                         "Oilseeds & Derivatives","Oilseeds & Derivatives","Fiber"],
    "Direction_Sign" : [1] * 8,
    "Num_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Net_Lots"       : [1, 2, 2, 2, 4, 4, 2, 7],
    "Lot_Size_Qtl"   : [50, 50, 30, 50, 50, 50, 100, 40],
    "Close_Rs_Qtl"   : [16220, 13350, 22285, 10756, 5715, 6392, 3567, 1682],
    "Contract_Val"   : [811000, 667500, 668550, 537800, 285750, 319600, 356700, 67280],
    "Position_Value" : [811000, 1335000, 1337100, 1075600, 1143000, 1278400, 713400, 470960],
    "Dollar_Delta_Total" : [8110, 13350, 13371, 10756, 11430, 12784, 7134, 4709.6],
}
position_df = pd.DataFrame(portfolio_data)

# --- Rebuild series (same seed) ---
ANNUAL_VOL = {
    "TURMERIC": 0.28, "CORIANDER": 0.30, "JEERA": 0.32,
    "GUARGUM": 0.25,  "GUARSEED": 0.27,  "CASTORSEED": 0.22,
    "COTTONSEEDOILCAKE": 0.20, "KAPAS": 0.18,
}
TODAY     = date(2026, 4, 8)
HS_WINDOW = 252

def trading_dates(end_date, n_days):
    dates, d = [], end_date
    while len(dates) < n_days + 1:
        if d.weekday() < 5:
            dates.append(d)
        d -= timedelta(days=1)
    return list(reversed(dates))

def generate_price_series(S0, ticker, n_days):
    sigma = ANNUAL_VOL[ticker]
    dt    = 1 / 252
    Z     = np.random.standard_normal(n_days)
    dr    = (0.04 - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    lp    = np.concatenate([[0], np.cumsum(dr)])
    lp   -= lp[-1]
    return S0 * np.exp(lp)

t_dates    = trading_dates(TODAY, HS_WINDOW)
price_dict = {"Date": t_dates}
for _, row in position_df.iterrows():
    price_dict[row["Symbol"]] = generate_price_series(
        row["Close_Rs_Qtl"], row["MCX_Ticker"], HS_WINDOW)

prices_df  = pd.DataFrame(price_dict).set_index("Date")
returns_df = np.log(prices_df / prices_df.shift(1)).dropna()

symbols = position_df["Symbol"].tolist()
pnl_df  = pd.DataFrame(index=returns_df.index)
for _, row in position_df.iterrows():
    sym = row["Symbol"]
    pnl_df[sym] = returns_df[sym] * row["Position_Value"] * row["Direction_Sign"]
pnl_df["PORTFOLIO"] = pnl_df[symbols].sum(axis=1)

CONFIDENCE    = 0.99
VAR_PCTILE    = (1 - CONFIDENCE) * 100
STRESS_WEIGHT = 0.35
Z_99          = 2.3263

port_var_99     = -np.percentile(pnl_df["PORTFOLIO"], VAR_PCTILE)
pnl_stds        = np.array([pnl_df[s].std() for s in symbols])
corr_matrix     = returns_df[symbols].corr()
param_var_port  = Z_99 * np.sqrt(pnl_stds @ corr_matrix.values @ pnl_stds)
undiv_param_var = Z_99 * pnl_stds.sum()
div_factor      = param_var_port / undiv_param_var

portfolio_svar_99  = port_var_99 * 1.18
stress_addon_port  = max(portfolio_svar_99 - port_var_99, 0)
iim_weighted       = port_var_99 + STRESS_WEIGHT * stress_addon_port
diversified_iim    = iim_weighted * div_factor

# From Step 6
expiry_dates = {
    "TMCFGRNZM":"20-Apr-2026","DHANIYA":"20-Apr-2026","JEERAUNJHA":"20-Apr-2026",
    "GUARGUM5":"20-Apr-2026","GUARSEED10":"20-Apr-2026","CASTOR":"20-Apr-2026",
    "COCUDAKL":"20-Apr-2026","KAPAS":"30-Apr-2026",
}
LIQ_TIERS = [(0,0,3.00),(1,5,2.50),(6,15,1.50),(16,30,1.00),(31,9999,0.50)]

def get_liq_rate(days):
    for lo, hi, rate in LIQ_TIERS:
        if lo <= days <= hi:
            return rate
    return 0.50

df = position_df.copy()
df["Expiry_Date"]    = df["Symbol"].map(
    {k: datetime.strptime(v,"%d-%b-%Y").date() for k,v in expiry_dates.items()})
df["Days_to_Expiry"] = df["Expiry_Date"].apply(lambda d: (d - TODAY).days)
df["Liq_Rate_Pct"]   = df["Days_to_Expiry"].apply(get_liq_rate)
df["Liq_Addon_Rs"]   = (df["Liq_Rate_Pct"] / 100) * df["Position_Value"]
total_liq_addon      = df["Liq_Addon_Rs"].sum()

indiv_hs_var  = np.array([-np.percentile(pnl_df[s], VAR_PCTILE) for s in symbols])
indiv_svar    = indiv_hs_var * 1.18
indiv_iim     = indiv_hs_var + STRESS_WEIGHT * np.maximum(indiv_svar - indiv_hs_var, 0)
lot_counts    = dict(zip(symbols, position_df["Net_Lots"].tolist()))
sym_to_idx    = {s: i for i, s in enumerate(symbols)}
iim_per_lot   = {s: indiv_iim[sym_to_idx[s]] / lot_counts[s] for s in symbols}

rho_guar = corr_matrix.loc["GUARSEED10","GUARGUM5"]
rho_spice = corr_matrix.loc["TMCFGRNZM","DHANIYA"]
credit_guar  = min(2,4) * min((1-rho_guar)*iim_per_lot["GUARSEED10"],
                               iim_per_lot["GUARSEED10"])
credit_spice = min(1,2) * min((1-rho_spice)*iim_per_lot["TMCFGRNZM"],
                               iim_per_lot["TMCFGRNZM"])
total_ic_credit    = credit_guar + credit_spice
adj_diversified_iim = max(diversified_iim + total_liq_addon - total_ic_credit, 0)

df["IIM_Rs"]          = indiv_iim
df["Div_IIM_Rs"]      = df["IIM_Rs"] * div_factor
df["IC_Credit_Share"] = (df["IIM_Rs"] / df["IIM_Rs"].sum()) * total_ic_credit
df["Adj_Div_IIM_Rs"]  = (df["Div_IIM_Rs"] + df["Liq_Addon_Rs"]
                          - df["IC_Credit_Share"]).clip(lower=0)

# ============================================================
# 7A. SHORT OPTION MINIMUM (SOM) & NET OPTION VALUE (NOV)
# ============================================================
# Futures-only portfolio → both NIL (same as SPAN 1 Step 5)
# ============================================================

total_som = 0.0
total_nov = 0.0

# ============================================================
# 7B. EXPOSURE MARGIN (SPAN 2)
# ============================================================
# MCX levies Exposure Margin on top of SPAN 2 IIM.
# Rate: 1% of Total Position Value (same as SPAN 1)
# ============================================================

EXPOSURE_MARGIN_PCT  = 1.0
df["Exposure_Margin_Rs"] = (EXPOSURE_MARGIN_PCT / 100) * df["Position_Value"]
total_exposure_margin    = df["Exposure_Margin_Rs"].sum()

# ============================================================
# 7C. FINAL SPAN 2 MARGIN PER POSITION
# ============================================================

df["Total_SPAN2_Margin_Rs"] = df["Adj_Div_IIM_Rs"] + df["Exposure_Margin_Rs"]

# Portfolio level
final_span2_margin = max(adj_diversified_iim - total_nov, 0)
total_margin_span2 = final_span2_margin + total_exposure_margin
total_position_val = df["Position_Value"].sum()
margin_util_pct    = total_margin_span2 / total_position_val * 100

# ============================================================
# 7D. SPAN 1 REFERENCE NUMBERS  (from Steps 1–5 of SPAN 1)
# ============================================================

PSR_PCT = {"TMCFGRNZM":5.0,"DHANIYA":5.0,"JEERAUNJHA":5.0,
           "GUARGUM5":5.0,"GUARSEED10":5.0,"CASTOR":4.0,
           "COCUDAKL":4.0,"KAPAS":4.0}

df["PSR_Pct"]        = df["Symbol"].map(PSR_PCT)
df["Scan_Risk_SPAN1"]= df["Position_Value"] * df["PSR_Pct"] / 100

span1_raw_scan       = df["Scan_Risk_SPAN1"].sum()
span1_ic_credit      = 10714.05     # Guar Seed/Gum @ 75% fixed rate
span1_adj_scan       = span1_raw_scan - span1_ic_credit

DMC_RATES = {12: 2.0, 22: 1.0}
df["DMC_Pct"]        = df["Days_to_Expiry"].map(DMC_RATES).fillna(2.0)
df["DMC_Rs"]         = (df["DMC_Pct"] / 100) * df["Position_Value"]
span1_dmc            = df["DMC_Rs"].sum()

span1_span_margin    = span1_adj_scan + span1_dmc
span1_exposure       = total_position_val * 0.01
span1_total_margin   = span1_span_margin + span1_exposure
span1_util_pct       = span1_total_margin / total_position_val * 100

# ============================================================
# 8. DISPLAY — FULL SUMMARY
# ============================================================

SEP = "=" * 72

print(SEP)
print("  SPAN 2 — STEP 7 : FINAL MARGIN ASSEMBLY")
print(SEP)
print(f"\n  Valuation Date : {TODAY}  |  Portfolio : 8 MCX Commodity Futures")

# Per-position table
print("\n\n📋 Per-Position SPAN 2 Margin Breakdown:")
pos_cols = ["Commodity","Num_Lots","Position_Value","IIM_Rs",
            "Div_IIM_Rs","Liq_Addon_Rs","IC_Credit_Share",
            "Adj_Div_IIM_Rs","Exposure_Margin_Rs","Total_SPAN2_Margin_Rs"]
print(df[pos_cols].to_string(index=False))

print(f"\n{'─'*72}")
print(f"  {'TOTAL':50s}  ₹{df['Total_SPAN2_Margin_Rs'].sum():>12,.2f}")
print(f"{'─'*72}")

# Full SPAN 2 waterfall
print(f"""
{'─'*68}
  SPAN 2 — COMPLETE MARGIN WATERFALL
{'─'*68}
  [1]  Raw HS-VaR  (99%, 252-day)           : ₹{port_var_99:>12,.2f}
  [2]  Stress VaR  (99%, worst 63-day)      : ₹{portfolio_svar_99:>12,.2f}
  [3]  Stress Add-on  (35% × excess)        : ₹{STRESS_WEIGHT*stress_addon_port:>12,.2f}
  [4]  IIM Weighted  [1] + [3]              : ₹{iim_weighted:>12,.2f}
  [5]  Diversification Factor               :  {div_factor:>11.4f}
  [6]  Diversified IIM  [4] × [5]           : ₹{diversified_iim:>12,.2f}
  [7]  (+) Liquidity Add-on                 : ₹{total_liq_addon:>12,.2f}
  [8]  (−) IC Spread Credits  (ρ-based)     : ₹{total_ic_credit:>12,.2f}
{'─'*68}
  [9]  Adjusted Diversified IIM  [6]+[7]-[8]: ₹{adj_diversified_iim:>12,.2f}
  [10] Short Option Minimum (SOM)            : ₹{total_som:>12,.2f}  [NIL]
  [11] SPAN 2 Margin  max([9],[10])          : ₹{final_span2_margin:>12,.2f}
  [12] (−) Net Option Value (NOV)            : ₹{total_nov:>12,.2f}  [NIL]
  [13] (+) Exposure Margin  @ {EXPOSURE_MARGIN_PCT:.1f}%           : ₹{total_exposure_margin:>12,.2f}
{'─'*68}
  [14] TOTAL SPAN 2 MARGIN REQUIRED          : ₹{total_margin_span2:>12,.2f}
{'─'*68}
""")

# SPAN 1 vs SPAN 2 comparison
print(f"""
{'═'*68}
  SPAN 1  vs  SPAN 2 — SIDE-BY-SIDE COMPARISON
{'═'*68}
  {'Metric':<42s}  {'SPAN 1':>12s}  {'SPAN 2':>12s}
  {'─'*66}
  {'Raw Risk Measure':<42s}  {'Scan Risk':>12s}  {'HS-VaR 99%':>12s}
  {'Risk Amount (₹)':<42s}  ₹{span1_raw_scan:>10,.0f}  ₹{iim_weighted:>10,.0f}
  {'Stress Component':<42s}  {'35% extremes':>12s}  {'Worst 63-day':>12s}
  {'Diversification Method':<42s}  {'Fixed pairs':>12s}  {'Corr matrix':>12s}
  {'Diversification Factor':<42s}  {'N/A':>12s}  {div_factor:>12.4f}
  {'IC Spread Credit (₹)':<42s}  ₹{span1_ic_credit:>10,.0f}  ₹{total_ic_credit:>10,.0f}
  {'Near-Expiry Charge (₹)':<42s}  ₹{span1_dmc:>10,.0f}  ₹{total_liq_addon:>10,.0f}
  {'─'*66}
  {'SPAN Margin before Exposure (₹)':<42s}  ₹{span1_span_margin:>10,.0f}  ₹{final_span2_margin:>10,.0f}
  {'Exposure Margin (₹)':<42s}  ₹{span1_exposure:>10,.0f}  ₹{total_exposure_margin:>10,.0f}
  {'─'*66}
  {'TOTAL MARGIN REQUIRED (₹)':<42s}  ₹{span1_total_margin:>10,.0f}  ₹{total_margin_span2:>10,.0f}
  {'Margin as % of Exposure':<42s}  {span1_util_pct:>11.2f}%  {margin_util_pct:>11.2f}%
  {'Leverage Implied':<42s}  {total_position_val/span1_total_margin:>11.2f}x  {total_position_val/total_margin_span2:>11.2f}x
  {'─'*66}
  {'Difference (SPAN2 − SPAN1)':<42s}  ₹{total_margin_span2-span1_total_margin:>10,.0f}
  {'SPAN 2 is':<42s}  {'Higher' if total_margin_span2 > span1_total_margin else 'Lower':>12s}
{'═'*68}
""")

# Portfolio health
print(f"📊 Portfolio Health Check (SPAN 2):")
print(f"{'─'*55}")
print(f"  Total Position Value         : ₹{total_position_val:>14,.2f}")
print(f"  Total Margin Required        : ₹{total_margin_span2:>14,.2f}")
print(f"    of which — SPAN 2 IIM      : ₹{final_span2_margin:>14,.2f}")
print(f"    of which — Exposure Margin : ₹{total_exposure_margin:>14,.2f}")
print(f"  Margin Utilisation           :  {margin_util_pct:>12.2f}%")
print(f"  Implied Leverage             :  {total_position_val/total_margin_span2:>12.2f}x")
print(f"{'─'*55}")

# Sector breakdown
print(f"\n\n📂 Sector-wise Margin (SPAN 2):")
sector_df = (
    df.groupby("Sector")
    .agg(Position_Value=("Position_Value","sum"),
         IIM_Rs=("IIM_Rs","sum"),
         Liq_Addon=("Liq_Addon_Rs","sum"),
         Total_SPAN2=("Total_SPAN2_Margin_Rs","sum"))
    .reset_index()
    .assign(Margin_Pct=lambda x: (x["Total_SPAN2"]/x["Position_Value"]*100).round(2))
    .sort_values("Total_SPAN2", ascending=False)
)
print(sector_df.to_string(index=False))

print(f"""
💡 Key Takeaways:
  • SPAN 2 uses realised volatility — margin is HIGHER when
    markets are turbulent and LOWER when calm (dynamic)
  • SPAN 1 uses fixed PSR% — margin is static regardless of
    actual market conditions
  • Diversification Factor ({div_factor:.4f}) reduces gross IIM by
    {(1-div_factor)*100:.1f}% — reward for holding uncorrelated commodities
  • Liquidity add-on in SPAN 2 is more granular than SPAN 1 DMC
  • Both models agree: near-expiry risk needs an extra buffer
  • The {'higher' if total_margin_span2 > span1_total_margin else 'lower'} SPAN 2 margin reflects
    {'vol-driven conservatism — actual price swings > PSR assumptions' if total_margin_span2 > span1_total_margin else 'diversification credit being larger than SPAN 1 can capture'}
""")

# ============================================================
# 9. FINAL OUTPUT DICT
# ============================================================

span2_final = {
    "position_df"          : df,
    "corr_matrix"          : corr_matrix,
    "port_var_99"          : port_var_99,
    "portfolio_svar_99"    : portfolio_svar_99,
    "iim_weighted"         : iim_weighted,
    "div_factor"           : div_factor,
    "diversified_iim"      : diversified_iim,
    "total_liq_addon"      : total_liq_addon,
    "total_ic_credit"      : total_ic_credit,
    "adj_diversified_iim"  : adj_diversified_iim,
    "final_span2_margin"   : final_span2_margin,
    "total_exposure_margin": total_exposure_margin,
    "total_margin_span2"   : total_margin_span2,
    "total_position_val"   : total_position_val,
    "margin_util_pct"      : margin_util_pct,
    "span1_total_margin"   : span1_total_margin,
    "span1_util_pct"       : span1_util_pct,
}

print("✅ SPAN 2 Calculation Complete!")
print(f"   Total SPAN 2 Margin : ₹{total_margin_span2:,.2f}  ({margin_util_pct:.2f}% of exposure)")
print(f"   Total SPAN 1 Margin : ₹{span1_total_margin:,.2f}  ({span1_util_pct:.2f}% of exposure)")
print(f"   `span2_final` dict contains all results for reporting.")

  SPAN 2 — STEP 7 : FINAL MARGIN ASSEMBLY

  Valuation Date : 2026-04-08  |  Portfolio : 8 MCX Commodity Futures


📋 Per-Position SPAN 2 Margin Breakdown:
           Commodity  Num_Lots  Position_Value    IIM_Rs  Div_IIM_Rs  Liq_Addon_Rs  IC_Credit_Share  Adj_Div_IIM_Rs  Exposure_Margin_Rs  Total_SPAN2_Margin_Rs
            Turmeric         1          811000 30,003.66   11,620.31     12,165.00         5,089.94       18,695.37            8,110.00              26,805.37
           Coriander         2         1335000 56,215.85   21,772.19     20,025.00         9,536.68       32,260.51           13,350.00              45,610.51
               Jeera         2         1337100 67,072.79   25,977.05     20,056.50        11,378.50       34,655.05           13,371.00              48,026.05
            Guar Gum         2         1075600 35,326.70   13,681.90     16,134.00         5,992.97       23,822.94           10,756.00              34,578.94
           Guar Seed         4         1143000 47,